# 06 · Mutability, Copying & Choosing the Right Data Structure

**Module 03 – Data Structures** · Estimated time: 40 min

### 🎯 What you will learn
- Mutable vs immutable objects, and why it matters
- Names, references and aliasing
- How arguments are passed to functions
- **Shallow copy vs deep copy**
- Hashability
- A decision guide: list vs tuple vs set vs dict (📘 *When to use each*)
- Time complexity of common operations

## 1. Mutable vs immutable

| Immutable (can't change in place) | Mutable (can change in place) |
|---|---|
| `int`, `float`, `complex`, `bool` | `list` |
| `str`, `bytes` | `dict` |
| `tuple`, `frozenset`, 🆕 `frozendict` | `set` |
| `None` | `bytearray`, most custom objects |

"Changing" an immutable object really creates a **new** object:

In [1]:
s = "hi"
print(id(s))
s += "!"           # a NEW string object; the name s is re-bound to it
print(id(s))

lst = [1, 2]
print(id(lst))
lst += [3]         # the SAME list is modified in place
print(id(lst))

4352396736
4456469376
4461802304
4461802304


## 2. Aliasing: two names, one object

In [2]:
a = [1, 2, 3]
b = a              # b is an alias, not a copy
b.append(4)
print(a)           # [1, 2, 3, 4]: surprise!
print(a is b)

[1, 2, 3, 4]
True


With immutable objects aliasing is harmless, because nobody can modify the shared object:

In [3]:
x = 10
y = x
y += 1
print(x, y)        # x is unaffected

10 11


## 3. Passing arguments to functions ("pass by object reference")

The function receives a reference to the **same object**. Mutating it inside the function affects the caller; re-binding the parameter does not.

In [4]:
def add_item(items):
    items.append("new")        # mutates the caller's list

def replace_list(items):
    items = ["replaced"]       # only re-binds the local name

my_items = ["a"]
add_item(my_items)
replace_list(my_items)
print(my_items)

['a', 'new']


## 4. Shallow copy vs deep copy

- **Shallow copy:** a new outer container, but the **inner objects are shared**.
  `lst.copy()`, `lst[:]`, `list(lst)`, `dict.copy()`, `copy.copy(x)`
- **Deep copy:** recursively copies everything. `copy.deepcopy(x)`

In [5]:
import copy

original = [[1, 2], [3, 4]]
shallow = original.copy()
deep = copy.deepcopy(original)

original[0].append(99)       # mutate an INNER list
original.append([5])         # mutate the OUTER list

print("original:", original)
print("shallow: ", shallow)  # sees the inner change, not the outer one
print("deep:    ", deep)     # completely independent

original: [[1, 2, 99], [3, 4], [5]]
shallow:  [[1, 2, 99], [3, 4]]
deep:     [[1, 2], [3, 4]]


```
original ──► [ ●, ● ]          shallow ──► [ ●, ● ]
               │  └──────────────────────────┼──┐
               └───────────► [1, 2, 99] ◄────┘  └─► [3, 4]
deep ──► [ ●, ● ] ──► its own [1, 2] and [3, 4]
```

> 💡 Rule of thumb: for flat lists of numbers/strings, a shallow copy is enough. For nested mutable structures, use `deepcopy`.

## 5. Hashability

An object is **hashable** if it has a hash value that never changes during its lifetime. Only hashable objects can be **dict keys** or **set items**.

In [6]:
for obj in [42, "text", (1, 2), frozenset({1}), frozendict(a=1), (1, [2]), [1], {1}, {"a": 1}]:
    try:
        hash(obj)
        print(f"{obj!r:<22} hashable")
    except TypeError:
        print(f"{obj!r:<22} NOT hashable")

42                     hashable
'text'                 hashable
(1, 2)                 hashable
frozenset({1})         hashable
frozendict({'a': 1})   hashable
(1, [2])               NOT hashable
[1]                    NOT hashable
{1}                    NOT hashable
{'a': 1}               NOT hashable


## 6. Choosing a data structure (📘 handbook + extended)

| Structure | Ordered | Mutable | Duplicates | Lookup by | Best for |
|---|---|---|---|---|---|
| `list` | ✅ | ✅ | ✅ | index | sequences that change; stacks |
| `tuple` | ✅ | ❌ | ✅ | index | fixed records, dict keys, return values |
| `set` | ❌ | ✅ | ❌ | value | uniqueness, membership tests, set math |
| `frozenset` | ❌ | ❌ | ❌ | value | sets as keys / set members |
| `dict` | ✅ (insertion) | ✅ | keys ❌ | key | fast lookups, structured data, counting |
| 🆕 `frozendict` | ✅ | ❌ | keys ❌ | key | constant config, dict as key |
| `collections.deque` | ✅ | ✅ | ✅ | ends | queues (fast at both ends) |
| `collections.Counter` | ✅ | ✅ | - | key | counting things |
| `dataclass` / `namedtuple` | - | ✅ / ❌ | - | attribute | records with named fields |

### Decision flow
1. Do you need to look things up by a **key/name**? → `dict`
2. Do you only care about **unique** items / membership? → `set`
3. Is the collection **fixed** (a record)? → `tuple` (or `namedtuple`/`dataclass`)
4. Otherwise → `list`

## 7. Time complexity cheat-sheet (average case)

| Operation | list | dict / set |
|---|---|---|
| index / key access | O(1) | O(1) |
| `x in container` | **O(n)** | **O(1)** |
| append / add | O(1) | O(1) |
| insert/pop at front | **O(n)** (use `deque`) | - |
| delete by value | O(n) | O(1) |
| sort | O(n log n) | - |

O(1) = constant time, O(n) = grows with size.

In [7]:
import timeit
n = 50_000
data_list = list(range(n))
data_set = set(data_list)
print("list in:", round(timeit.timeit(lambda: n - 1 in data_list, number=500), 4), "s")
print("set  in:", round(timeit.timeit(lambda: n - 1 in data_set, number=500), 4), "s")

list in: 0.1186 s
set  in: 0.0 s


## 📝 Summary
- Immutable: numbers, str, tuple, frozenset, frozendict. Mutable: list, dict, set.
- `=` never copies; it creates another reference.
- Use `copy()` for shallow and `copy.deepcopy()` for deep copies.
- Only hashable objects can be dict keys or set members.
- Pick the structure by meaning: key lookup → dict, uniqueness → set, fixed → tuple, otherwise list.

## 🏋️ Practice
1. Predict the output:
   ```python
   a = [1, [2, 3]]
   b = a[:]
   b[1].append(4)
   b.append(5)
   print(a)
   ```
2. Write a function `safe_append(lst, item)` that returns a **new** list with the item and leaves the original unchanged.
3. Which structure would you choose for: (a) a phone book, (b) RGB colour, (c) visited URLs, (d) a to-do list?

<details><summary>Show solutions</summary>

```python
# 1 → [1, [2, 3, 4]]  (inner list is shared, the outer append isn't)

def safe_append(lst, item):
    return lst + [item]          # or [*lst, item]

# 3 → (a) dict  (b) tuple  (c) set  (d) list
```
</details>